In [ ]:
from transformers import pipeline
model = pipeline("sentiment-analysis")
print(model("I love this project"))
print(model("This is terrible"))
print(model("I'm not sure how I feel about this"))

import requests
API_KEY = "8afa2742ee464a1493576c11d7f68e02"
url = "https://newsapi.org/v2/everything"
params = {
    "q": "finance AND (stock market OR investment)",
    "language": "en",
    "apiKey": API_KEY
}
response = requests.get(url, params=params)
data = response.json()

articles = data.get("articles", [])
news_list = []
for article in articles:
    text = str(article.get("title", "")) + " " + str(article.get("description", ""))
    news_list.append(text)
print(news_list[:3])

# To get more articles
import requests
from datetime import datetime, timedelta

API_KEY = "8afa2742ee464a1493576c11d7f68e02"
url = "https://newsapi.org/v2/everything"

# Query parameters
query = "finance AND (stock market OR investment)"
language = "en"
page_size = 100  # max per request

# Date range setup: last 30 days
end_date = datetime.today()
start_date = end_date - timedelta(days=30)
all_articles = []

# Split the 30-day range into 3-day chunks (adjust as needed)
delta = timedelta(days=3)
current_start = start_date

while current_start < end_date:
    current_end = min(current_start + delta, end_date)
    page = 1
    while True:
        params = {
            "q": query,
            "language": language,
            "pageSize": page_size,
            "page": page,
            "from": current_start.strftime("%Y-%m-%d"),
            "to": current_end.strftime("%Y-%m-%d"),
            "apiKey": API_KEY
        }
        response = requests.get(url, params=params)
        data = response.json()
        if "articles" in data and data["articles"]:
            all_articles.extend(data["articles"])
            if len(data["articles"]) < page_size:
                break  # No more pages for this date range
            page += 1
        else:
            break  # No articles for this page/date range
    current_start += delta

# Extract text
news_list = []
for article in all_articles:
    text = str(article.get("title", "")) + " " + str(article.get("description", ""))
    news_list.append(text)
print(f"Total articles fetched: {len(news_list)}")
print(news_list[:10])

results = []
for news in news_list:
    result = model(news[:512])[0]
    results.append(result)
print(results[:7])

# --- Directly count POSITIVE and NEGATIVE without mapping ---
import json
sentiment_counts = {"POSITIVE": 0, "NEGATIVE": 0}

for r in results:
    label = r["label"]
    if label in sentiment_counts:
        sentiment_counts[label] += 1

with open("results_file2.json", "w") as f:
    json.dump(sentiment_counts, f, indent=4)

with open("results.json", "w") as f:
    json.dump(sentiment_counts, f, indent=4)
# -------------------------------------------------------------

import pandas as pd
df = pd.DataFrame(results)
print(df.head())

import matplotlib.pyplot as plt
if not df.empty and 'label' in df.columns:
    df['label'].value_counts().plot(kind='bar')
    plt.title("News Sentiment Analysis")
    plt.xlabel("Sentiment")
    plt.ylabel("Count")
    plt.show()